# Pythia-160M TopK SAE seed-stability prototype

**Question:** when only SAE initialization changes, how much of the learned dictionary is shared?

| Fixed item | Value |
|---|---|
| Model / site | `pythia-160m`, `blocks.6.hook_mlp_out` |
| Activation width | `d_in=768` |
| SAE | TopK, `d_sae=32_768`, `k=32` |
| Paper scale | first 8B Pile tokens; 2 seeds first, then 9 |
| Primary metric | encoder + decoder Hungarian matching; shared if both choose the same feature and both cosine similarities are `>=0.7` |

> The paper used EleutherAI's `sae` now known as `sparsify`; this is an SAELens port. The compatible Pile dataset below is suitable for prototyping, but a strict replication must recover the exact token stream, ordering, and remaining optimizer settings.

In [1]:
%pip install -q "sae-lens==6.46.0" scipy python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 17.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.2/312.2 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 167.3 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.4/298.4 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.9/236.9 kB 26.5 MB/s eta 0:00:00


## Colab storage and credentials

Mount Google Drive, then load `HF_TOKEN`, `HF_REPO_ID`, and `WANDB_API_KEY` from `MyDrive/SAE/.env` without displaying their values. Checkpoints and final SAEs use the same Drive mount.

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

ENV_PATH = Path("/content/drive/MyDrive/SAE/.env")
if not load_dotenv(ENV_PATH, override=True):
    raise FileNotFoundError(f"Could not load {ENV_PATH}")

REQUIRED_ENV_VARS = ("HF_TOKEN", "HF_REPO_ID", "WANDB_API_KEY")
missing = [name for name in REQUIRED_ENV_VARS if not os.getenv(name)]
if missing:
    raise RuntimeError(f"Missing environment variables in {ENV_PATH}: {missing}")

print("Google Drive mounted and credentials loaded.")

Mounted at /content/drive
Google Drive mounted and credentials loaded.


## 1. Freeze the experiment

Use one shared activation stream for every SAE. Do **not** run separate trainers with different runner seeds: that can also change data order. Set `RUN_MODE` to `smoke`, `tutorial` (30K steps), or `paper` (8B tokens).

In [3]:
from pathlib import Path

import torch
from sae_lens import (
    LoggingConfig,
    MultiSAETrainingRunner,
    MultiSAETrainingRunnerConfig,
    TopKTrainingSAE,
    TopKTrainingSAEConfig,
)

SEEDS = [0, 1]
BATCH_TOKENS = 4096
RUN_MODE = "smoke"  # smoke | tutorial | paper
TRAINING_TOKENS = {
    "smoke": 2_000_000,
    "tutorial": 30_000 * BATCH_TOKENS,
    "paper": 8_000_000_000,
}[RUN_MODE]
TOTAL_TRAINING_STEPS = TRAINING_TOKENS // BATCH_TOKENS
LR_WARM_UP_STEPS = 0
LR_DECAY_STEPS = TOTAL_TRAINING_STEPS // 5
DATA_SEED = 42

LOG_TO_WANDB = bool(os.getenv("WANDB_API_KEY"))
WANDB_PROJECT = "pythia-160m-seeds"
WANDB_ENTITY = None  # set to your W&B team, if needed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HOOK = "blocks.6.hook_mlp_out"
OUT = Path("/content/drive/MyDrive/SAE/research-1")
OUT.mkdir(parents=True, exist_ok=True)

## 2. Initialize SAEs, then train on the same batches

`override_saes` gives each dictionary a recorded initialization seed while `MultiSAETrainingRunner` sends every dictionary identical activation batches. TopK uses `k` instead of the tutorial's `l1_*` settings, and the multi-runner activation store detects the tokenized dataset column. W&B logging is enabled automatically after the Drive `.env` cell succeeds.

In [5]:
print("Device:", DEVICE)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name())

Device: cuda
CUDA: True
NVIDIA A100-SXM4-80GB


In [4]:
sae_cfgs = {
    f"seed_{seed}": TopKTrainingSAEConfig(
        d_in=768,
        d_sae=32_768,
        k=32,
        apply_b_dec_to_input=False,
        normalize_activations="expected_average_only_in",
    )
    for seed in SEEDS
}

initialized_saes = {}
for seed in SEEDS:
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    initialized_saes[f"seed_{seed}"] = TopKTrainingSAE(sae_cfgs[f"seed_{seed}"])

runner_cfg = MultiSAETrainingRunnerConfig(
    saes=sae_cfgs,
    hook_names=HOOK,
    model_name="pythia-160m",
    dataset_path="apollo-research/monology-pile-uncopyrighted-tokenizer-EleutherAI-gpt-neox-20b",
    streaming=True,
    context_size=512,
    training_tokens=TRAINING_TOKENS,
    train_batch_size_tokens=BATCH_TOKENS,
    store_batch_size_prompts=16,
    n_batches_in_buffer=64,
    lr=5e-5,
    adam_beta1=0.9,
    adam_beta2=0.999,
    lr_scheduler_name="constant",
    lr_warm_up_steps=LR_WARM_UP_STEPS,
    lr_decay_steps=LR_DECAY_STEPS,
    feature_sampling_window=1000,
    dead_feature_window=1000,
    dead_feature_threshold=1e-4,
    device=DEVICE,
    seed=DATA_SEED,
    dtype="float32",
    autocast=DEVICE == "cuda",
    autocast_lm=DEVICE == "cuda",
    n_checkpoints=2,
    save_final_checkpoint=True,
    checkpoint_path=str(OUT / "checkpoints"),
    output_path=str(OUT / "trained_saes"),
    n_batches_for_norm_estimate=100 if RUN_MODE == "smoke" else 1000,
    logger=LoggingConfig(
        log_to_wandb=LOG_TO_WANDB,
        wandb_project=WANDB_PROJECT,
        wandb_entity=WANDB_ENTITY,
        run_name=f"pythia-topk-{RUN_MODE}-{len(SEEDS)}-seeds-1", # Keep incrementing number at the end of the wandb run name.
        wandb_log_frequency=30,
        eval_every_n_wandb_logs=20,
    ),
)

runner = MultiSAETrainingRunner(
    runner_cfg, override_saes=initialized_saes
)
trained_saes = runner.run()

config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/375M [00:00<?, ?B/s]

KeyboardInterrupt: 

## 3. Upload completed SAEs

The Drive `.env` supplies `HF_TOKEN` and `HF_REPO_ID`. SAELens creates a public repository if it does not exist; create it beforehand for a private repository. This uploads inference weights/configuration for later `SAE.from_pretrained(...)` use. Resumable optimizer checkpoints remain on Drive.

In [ ]:
# import os

# from sae_lens import upload_saes_to_huggingface

# HF_REPO_ID = os.getenv("HF_REPO_ID")
# if HF_REPO_ID:
#     upload_saes_to_huggingface(
#         saes_dict=trained_saes,
#         hf_repo_id=HF_REPO_ID,
#         hf_revision=os.getenv("HF_REVISION", "main"),
#     )
#     print(f"Uploaded {list(trained_saes)} to https://huggingface.co/{HF_REPO_ID}")
# else:
#     print("Upload skipped: set HF_REPO_ID and HF_TOKEN on the remote machine.")

## 4. Quality gate before comparing seeds

For each seed, record reconstruction MSE / explained variance, CE loss recovered, `L0` (approximately 32), feature-density histogram, and dead-feature fraction. Compare overlap only when both runs have similar quality; otherwise seed effects are confounded with training failure.

## 5. Reproduce the paper's feature overlap

Match encoder and decoder directions separately. At 32K features, one dense cosine matrix is about 4 GiB in float32, and SciPy needs additional host memory. Run encoder and decoder matching sequentially on a high-memory machine.

In [ ]:
# import gc

# import torch.nn.functional as F
# from scipy.optimize import linear_sum_assignment

# def hungarian_match(a, b):
#     score = (F.normalize(a.float(), dim=1) @ F.normalize(b.float(), dim=1).T).cpu()
#     rows, cols = linear_sum_assignment(-score.numpy())
#     matched = score[rows, cols]
#     del score
#     gc.collect()
#     return torch.as_tensor(cols), matched

# def shared_fraction(sae_a, sae_b, threshold=0.7):
#     enc_idx, enc_sim = hungarian_match(sae_a.W_enc.detach().T, sae_b.W_enc.detach().T)
#     dec_idx, dec_sim = hungarian_match(sae_a.W_dec.detach(), sae_b.W_dec.detach())
#     shared = (enc_idx == dec_idx) & (enc_sim >= threshold) & (dec_sim >= threshold)
#     return {
#         "shared_fraction": shared.float().mean().item(),
#         "mean_encoder_cosine": enc_sim.mean().item(),
#         "mean_decoder_cosine": dec_sim.mean().item(),
#     }

# result = shared_fraction(trained_saes["seed_0"], trained_saes["seed_1"])
# result

## 6. Extend only after the baseline works

1. Train 9 seeds and compute all 36 pairwise matches.
2. Compare matched-feature activation overlap on one fixed held-out token set.
3. Add linear CKA and SVCCA on held-out feature-activation matrices; these measure representation-level similarity, not one-to-one feature recovery.
4. Sweep one factor at a time: `k`, dictionary width, tokens, layer/site, then SAE architecture.

**References:** [seed-stability paper](https://arxiv.org/abs/2501.16615), [released analysis/checkpoints](https://github.com/EleutherAI/sae_overlap), [feature-consistency position paper](https://arxiv.org/abs/2505.20254), [SAELens training guide](https://decoderesearch.github.io/SAELens/training_saes/), and local `references/sparse-autoencoders.md`.